# Convolutional Neural Networks — Implementations

Every layer twice more: on tensors with autograd supplying the backward pass, and through `torch.nn` with the notebook's own NumPy weight draw copied in. The layer fixtures compare a forward output and a backward gradient for identical weights and inputs; the full CNN reproduces the notebook's 30-epoch training run — same data draws, same batches, same learning rate — number for number.

## 14_conv2d

Sliding dot products, learned.

### torch

The same layer with `F.conv2d` doing the sliding window and autograd doing the calculus: one `_Y.backward(dY)` call produces the `dW`, `db` and `dX` the notebook derives by hand. `F.conv2d` computes cross-correlation — exactly what the scratch `forward` does — so no kernel flip appears anywhere. **What torch adds:** the entire backward pass for free, and a vectorised forward in place of four nested loops.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# hints:
# 1. F.conv2d cross-correlates — exactly what the scratch forward does; no flip needed.
# 2. Draw W with the caller's NumPy rng, then as_tensor it: torch's RNG never matches.
# 3. Make X a leaf with requires_grad_ in forward; _Y.backward(dY) then fills X.grad.
# 4. One backward call yields dW, db and dX — the three loops the notebook writes out.
# 5. Update under torch.no_grad(), then set .grad = None so the next step starts clean.


class Conv2D:
    """2D convolutional layer on tensors — autograd supplies the backward pass.

    Input shape:  (batch, C_in, H, W)
    Output shape: (batch, C_out, H_out, W_out)
    """

    def __init__(self, c_in, c_out, kernel_size, rng):
        k = kernel_size
        # He initialization, drawn with NumPy so every lane starts identically
        scale = np.sqrt(2.0 / (c_in * k * k))
        self.W = torch.as_tensor(rng.normal(0, scale, size=(c_out, c_in, k, k)))
        self.W.requires_grad_(True)
        self.b = torch.zeros(c_out, dtype=torch.float64, requires_grad=True)
        self.kernel_size = k
        self.X_cache = None
        self.dW = None
        self.db = None

    def forward(self, X):
        """Forward pass: F.conv2d is cross-correlation, like the scratch loop."""
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        self._Y = F.conv2d(X, self.W, self.b)
        return self._Y.detach()

    def backward(self, dY, lr):
        """Backward pass: one autograd call produces dW, db and dX."""
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        self.dW = self.W.grad.detach().clone()
        self.db = self.b.grad.detach().clone()
        dX = self.X_cache.grad.detach().clone()
        with torch.no_grad():
            self.W -= lr * self.dW
            self.b -= lr * self.db
        self.W.grad = None
        self.b.grad = None
        return dX


In [ ]:
# exports: Y_conv, dX_conv, dW_conv, db_conv, W_step
_rng_eq = np.random.default_rng(1414)
X_conv_eq = _rng_eq.normal(size=(2, 2, 6, 6))
dY_conv_eq = _rng_eq.normal(size=(2, 3, 4, 4))
_conv_eq = Conv2D(c_in=2, c_out=3, kernel_size=3, rng=np.random.default_rng(77))
Y_conv = _conv_eq.forward(X_conv_eq)
dX_conv = _conv_eq.backward(dY_conv_eq, lr=0.1)
dW_conv = _conv_eq.dW.clone()
db_conv = _conv_eq.db.clone()
W_step = _conv_eq.W.detach().clone()
print("forward:", tuple(Y_conv.shape), "-> dX:", tuple(dX_conv.shape))


In [ ]:
# An asymmetric kernel tells cross-correlation from convolution.
_k_eq = torch.zeros(1, 1, 3, 3, dtype=torch.float64)
_k_eq[0, 0, 1, 0], _k_eq[0, 0, 1, 2] = 1.0, -1.0
_probe = Conv2D(c_in=1, c_out=1, kernel_size=3, rng=np.random.default_rng(0))
_probe.W = _k_eq
_probe.b = torch.zeros(1, dtype=torch.float64)
_img_eq = torch.as_tensor(np.random.default_rng(3).normal(size=(1, 1, 5, 5)))
_out_eq = _probe.forward(_img_eq)
assert torch.allclose(_out_eq[0, 0], _img_eq[0, 0, 1:4, 0:3] - _img_eq[0, 0, 1:4, 2:5],
                      atol=1e-12), "forward is cross-correlation: no kernel flip"

# The bias gradient is the upstream gradient, summed per filter.
assert torch.allclose(db_conv, torch.as_tensor(dY_conv_eq).sum(dim=(0, 2, 3)),
                      atol=1e-12), "db = sum of dY over batch and positions"

# backward applied exactly one SGD step: W_after = W_before − lr·dW.
_W0_eq = Conv2D(c_in=2, c_out=3, kernel_size=3, rng=np.random.default_rng(77)).W.detach()
assert torch.allclose(W_step, _W0_eq - 0.1 * dW_conv, atol=1e-12), "one plain SGD step"
assert tuple(Y_conv.shape) == (2, 3, 4, 4), "valid padding: 6 − 3 + 1 = 4"


### library

`nn.Conv2d` with the notebook's He draw copied straight in — its weight layout is `(c_out, c_in, k, k)`, the same as the scratch layer's, and its forward is the same cross-correlation. **What the library adds:** a `Module` that owns its parameters, one optimiser step away from a real training loop.

In [ ]:
import numpy as np
import torch

# hints:
# 1. nn.Conv2d stores weight as (c_out, c_in, k, k) — the notebook's layout already.
# 2. .double() the module: float32 defaults would eat half the digits of agreement.
# 3. copy_ the NumPy draw under no_grad; zero the bias, which torch randomises.
# 4. The module's forward is F.conv2d — the same cross-correlation as the scratch loop.


class Conv2D:
    """nn.Conv2d behind the notebook's interface, weights copied from the same draw."""

    def __init__(self, c_in, c_out, kernel_size, rng):
        k = kernel_size
        scale = np.sqrt(2.0 / (c_in * k * k))
        self.layer = torch.nn.Conv2d(c_in, c_out, k).double()
        with torch.no_grad():
            W0 = rng.normal(0, scale, size=(c_out, c_in, k, k))
            self.layer.weight.copy_(torch.as_tensor(W0))
            self.layer.bias.zero_()
        self.kernel_size = k
        self.W = self.layer.weight.detach().clone()
        self.b = self.layer.bias.detach().clone()
        self.X_cache = None
        self.dW = None
        self.db = None

    def forward(self, X):
        """Forward pass through the module, input kept as a leaf for dX."""
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        self._Y = self.layer(X)
        return self._Y.detach()

    def backward(self, dY, lr):
        """Backward through the module, then a manual SGD step on its parameters."""
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        self.dW = self.layer.weight.grad.detach().clone()
        self.db = self.layer.bias.grad.detach().clone()
        dX = self.X_cache.grad.detach().clone()
        with torch.no_grad():
            self.layer.weight -= lr * self.dW
            self.layer.bias -= lr * self.db
        self.layer.zero_grad()
        self.W = self.layer.weight.detach().clone()
        self.b = self.layer.bias.detach().clone()
        return dX


In [ ]:
# exports: Y_conv, dX_conv, dW_conv, db_conv, W_step
_rng_eq = np.random.default_rng(1414)
X_conv_eq = _rng_eq.normal(size=(2, 2, 6, 6))
dY_conv_eq = _rng_eq.normal(size=(2, 3, 4, 4))
_conv_eq = Conv2D(c_in=2, c_out=3, kernel_size=3, rng=np.random.default_rng(77))
Y_conv = _conv_eq.forward(X_conv_eq)
dX_conv = _conv_eq.backward(dY_conv_eq, lr=0.1)
dW_conv = _conv_eq.dW.clone()
db_conv = _conv_eq.db.clone()
W_step = _conv_eq.W.detach().clone()
print("forward:", tuple(Y_conv.shape), "-> dX:", tuple(dX_conv.shape))


In [ ]:
assert tuple(_conv_eq.layer.weight.shape) == (3, 2, 3, 3), \
    "nn.Conv2d keeps (c_out, c_in, k, k) — no transposition needed"
assert torch.allclose(db_conv, torch.as_tensor(dY_conv_eq).sum(dim=(0, 2, 3)),
                      atol=1e-12), "db = sum of dY over batch and positions"
_W0_eq = Conv2D(c_in=2, c_out=3, kernel_size=3, rng=np.random.default_rng(77)).W
assert torch.allclose(W_step, _W0_eq - 0.1 * dW_conv, atol=1e-12), "one plain SGD step"


## 14_maxpool2d

Keep the strongest response in each window.

### torch

Max over non-overlapping windows via a reshape and `amax`; autograd routes the upstream gradient back to each window's maximum, which is the mask bookkeeping the scratch class does by hand. On an exact tie `amax` splits the gradient where the scratch mask gives it all to the first maximum — invisible on continuous inputs. **What torch adds:** the routing comes out of the graph instead of a stored mask.

In [ ]:
import numpy as np
import torch

# hints:
# 1. reshape to (batch, C, H//p, p, W//p, p), then amax over the two window axes.
# 2. Autograd routes the gradient back to each window's max — the mask is free.
# 3. On an exact tie amax splits the gradient; the scratch mask picks the first max.
# 4. Continuous random inputs never tie, which is what keeps the lanes comparable.


class MaxPool2D:
    """Max over non-overlapping windows; autograd remembers where the max was."""

    def __init__(self, pool_size=2):
        self.pool_size = pool_size
        self.X_cache = None

    def forward(self, X):
        """Forward: reshape into windows and take amax — no explicit mask."""
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        batch, C, H, W = X.shape
        p = self.pool_size
        windows = X.reshape(batch, C, H // p, p, W // p, p)
        self._Y = windows.amax(dim=(3, 5))
        return self._Y.detach()

    def backward(self, dY, lr=0.0):
        """Backward: autograd routes dY to the argmax of each window."""
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        return self.X_cache.grad.detach().clone()


In [ ]:
# exports: Y_pool_out, dX_pool_out
_rng_eq = np.random.default_rng(1442)
X_pool_eq = _rng_eq.normal(size=(2, 3, 6, 6))
dY_pool_eq = _rng_eq.normal(size=(2, 3, 3, 3))
_pool_eq = MaxPool2D(pool_size=2)
Y_pool_out = _pool_eq.forward(X_pool_eq)
dX_pool_out = _pool_eq.backward(dY_pool_eq)
print("pooled:", tuple(Y_pool_out.shape), "-> dX:", tuple(dX_pool_out.shape))


In [ ]:
# The notebook's own 4x4 example, replayed.
_probe = MaxPool2D(pool_size=2)
_Y_probe = _probe.forward(np.array([[[[1, 3, 2, 4],
                                      [5, 6, 7, 8],
                                      [3, 2, 1, 0],
                                      [1, 2, 3, 4]]]], dtype=float))
assert torch.allclose(_Y_probe, torch.tensor([[[[6.0, 8.0], [3.0, 4.0]]]],
                                             dtype=torch.float64)), "max of each 2x2 window"

# Gradient is conserved: the window sums of dX reproduce dY exactly.
_win_sums = dX_pool_out.reshape(2, 3, 3, 2, 3, 2).sum(dim=(3, 5))
assert torch.allclose(_win_sums, torch.as_tensor(dY_pool_eq), atol=1e-12), \
    "each window's dX sums to its dY"

# With no ties, exactly one position per window receives gradient.
assert int((dX_pool_out != 0).sum()) == 2 * 3 * 3 * 3, "one winner per window"


### library

`nn.MaxPool2d(2)` — kernel 2, stride 2, the same non-overlapping windows, and on ties it routes to the first maximum exactly like the scratch mask. There are no parameters to copy, so the lanes agree to machine precision. **What the library adds:** only a name — pooling is pure routing.

In [ ]:
import numpy as np
import torch

# hints:
# 1. nn.MaxPool2d(2) means kernel 2, stride 2 — the same non-overlapping windows.
# 2. Pooling has no parameters, so there is nothing to copy and nothing to drift.
# 3. Cache the leaf input; after _Y.backward(dY) its .grad is the routed gradient.


class MaxPool2D:
    """nn.MaxPool2d behind the scratch interface — pure routing, no parameters."""

    def __init__(self, pool_size=2):
        self.pool_size = pool_size
        self.layer = torch.nn.MaxPool2d(pool_size)
        self.X_cache = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        self._Y = self.layer(X)
        return self._Y.detach()

    def backward(self, dY, lr=0.0):
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        return self.X_cache.grad.detach().clone()


In [ ]:
# exports: Y_pool_out, dX_pool_out
_rng_eq = np.random.default_rng(1442)
X_pool_eq = _rng_eq.normal(size=(2, 3, 6, 6))
dY_pool_eq = _rng_eq.normal(size=(2, 3, 3, 3))
_pool_eq = MaxPool2D(pool_size=2)
Y_pool_out = _pool_eq.forward(X_pool_eq)
dX_pool_out = _pool_eq.backward(dY_pool_eq)
print("pooled:", tuple(Y_pool_out.shape), "-> dX:", tuple(dX_pool_out.shape))


In [ ]:
_probe = MaxPool2D(pool_size=2)
_Y_probe = _probe.forward(np.array([[[[1, 3, 2, 4],
                                      [5, 6, 7, 8],
                                      [3, 2, 1, 0],
                                      [1, 2, 3, 4]]]], dtype=float))
assert torch.allclose(_Y_probe, torch.tensor([[[[6.0, 8.0], [3.0, 4.0]]]],
                                             dtype=torch.float64)), "max of each 2x2 window"
_win_sums = dX_pool_out.reshape(2, 3, 3, 2, 3, 2).sum(dim=(3, 5))
assert torch.allclose(_win_sums, torch.as_tensor(dY_pool_eq), atol=1e-12), \
    "gradient conservation per window"


## 14_dense_layers

The classifier head: mask, reshape, affine map.

### torch

`ReLULayer` and `Flatten` port verbatim — a mask and a reshape are the same in any framework — while `DenseLayer` lets one `backward` call produce `dW`, `db` and `dX`. The weights still come from the notebook's NumPy `rng`, so every lane starts from identical numbers. **What torch adds:** the chain rule for the one layer that actually has parameters.

In [ ]:
import numpy as np
import torch

# hints:
# 1. ReLU's mask and Flatten's reshape port verbatim — only DenseLayer needs autograd.
# 2. He init still comes from the NumPy rng, as_tensor'd, so every lane matches.
# 3. DenseLayer.forward caches a leaf X; backward reads X.grad after _Y.backward(dY).
# 4. Subtract lr*grad under no_grad, then set .grad = None before the next step.


class ReLULayer:
    """Element-wise ReLU — the mask is the same arithmetic on tensors."""

    def __init__(self):
        self.mask = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64)
        self.mask = (X > 0).to(X.dtype)
        return X * self.mask

    def backward(self, dY, lr=0.0):
        return torch.as_tensor(dY, dtype=torch.float64) * self.mask


class Flatten:
    """Flatten spatial dimensions: (batch, C, H, W) → (batch, C*H*W)."""

    def __init__(self):
        self.input_shape = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64)
        self.input_shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dY, lr=0.0):
        return torch.as_tensor(dY, dtype=torch.float64).reshape(self.input_shape)


class DenseLayer:
    """Fully connected layer Y = X @ W + b, with autograd doing the calculus."""

    def __init__(self, d_in, d_out, rng):
        scale = np.sqrt(2.0 / d_in)
        self.W = torch.as_tensor(rng.normal(0, scale, size=(d_in, d_out)))
        self.W.requires_grad_(True)
        self.b = torch.zeros(d_out, dtype=torch.float64, requires_grad=True)
        self.X_cache = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        self._Y = X @ self.W + self.b
        return self._Y.detach()

    def backward(self, dY, lr):
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        dX = self.X_cache.grad.detach().clone()
        with torch.no_grad():
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
        self.W.grad = None
        self.b.grad = None
        return dX


In [ ]:
# exports: Y_dense, dX_dense, W_dense_step, b_dense_step
_rng_eq = np.random.default_rng(1456)
X_dense_eq = _rng_eq.normal(size=(4, 2, 4, 4))
dY_dense_eq = _rng_eq.normal(size=(4, 3))
_relu_eq = ReLULayer()
_flat_eq = Flatten()
_fc_eq = DenseLayer(32, 3, rng=np.random.default_rng(88))
Y_dense = _fc_eq.forward(_flat_eq.forward(_relu_eq.forward(X_dense_eq)))
dX_dense = _relu_eq.backward(_flat_eq.backward(_fc_eq.backward(dY_dense_eq, lr=0.1)))
W_dense_step = _fc_eq.W.detach().clone()
b_dense_step = _fc_eq.b.detach().clone()
print("head out:", tuple(Y_dense.shape), "-> dX:", tuple(dX_dense.shape))


In [ ]:
# The whole chain rule, written out with the pre-update weights.
_X_t = torch.as_tensor(X_dense_eq)
_W0_eq = DenseLayer(32, 3, rng=np.random.default_rng(88)).W.detach()
_expect = (torch.as_tensor(dY_dense_eq) @ _W0_eq.T).reshape(4, 2, 4, 4) * (_X_t > 0)
assert torch.allclose(dX_dense, _expect, atol=1e-12), \
    "dX = (dY @ W.T) reshaped, masked by the ReLU"

# b started at zero, so one step lands exactly at −lr·db with db = dY.sum(0).
assert torch.allclose(b_dense_step, -0.1 * torch.as_tensor(dY_dense_eq).sum(dim=0),
                      atol=1e-12), "one SGD step on the bias"

# Flatten is lossless: backward(forward(X)) restores shape and values.
_flat_rt = Flatten()
assert torch.allclose(_flat_rt.backward(_flat_rt.forward(_X_t)), _X_t), "Flatten round-trips"

# Dead units stay dead: no gradient flows where the input was negative.
assert float(dX_dense[_X_t < 0].abs().max()) == 0.0, "ReLU blocks gradient at X < 0"


### library

`nn.ReLU`, `nn.Flatten` and `nn.Linear` behind the scratch interface. `nn.Linear` stores its weight as `(d_out, d_in)` and computes `X @ W.T + b`, so the notebook's `(d_in, d_out)` draw is copied in transposed — the classic layout difference between hand-rolled and library dense layers. **What the library adds:** modules that own their parameters, plus that transposition to know about.

In [ ]:
import numpy as np
import torch

# hints:
# 1. nn.Linear stores weight as (d_out, d_in): copy the notebook's (d_in, d_out) draw .T.
# 2. Zero the module's bias — the notebook starts at zero, torch randomises it.
# 3. X @ W + b and Linear's X @ W.T + b are the same map once the copy is transposed.
# 4. Autograd through the module gives dX and the parameter grads in one call.


class ReLULayer:
    """nn.ReLU with autograd doing the mask bookkeeping."""

    def __init__(self):
        self.layer = torch.nn.ReLU()
        self.X_cache = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        self._Y = self.layer(X)
        return self._Y.detach()

    def backward(self, dY, lr=0.0):
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        return self.X_cache.grad.detach().clone()


class Flatten:
    """nn.Flatten — a reshape with the input shape cached for the way back."""

    def __init__(self):
        self.layer = torch.nn.Flatten()
        self.input_shape = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64)
        self.input_shape = X.shape
        return self.layer(X)

    def backward(self, dY, lr=0.0):
        return torch.as_tensor(dY, dtype=torch.float64).reshape(self.input_shape)


class DenseLayer:
    """nn.Linear with the notebook's (d_in, d_out) draw copied in transposed."""

    def __init__(self, d_in, d_out, rng):
        scale = np.sqrt(2.0 / d_in)
        self.layer = torch.nn.Linear(d_in, d_out).double()
        with torch.no_grad():
            W0 = rng.normal(0, scale, size=(d_in, d_out))
            self.layer.weight.copy_(torch.as_tensor(W0).T)
            self.layer.bias.zero_()
        self.W = self.layer.weight.detach().T.clone()
        self.b = self.layer.bias.detach().clone()
        self.X_cache = None

    def forward(self, X):
        X = torch.as_tensor(X, dtype=torch.float64).detach().clone().requires_grad_(True)
        self.X_cache = X
        self._Y = self.layer(X)
        return self._Y.detach()

    def backward(self, dY, lr):
        self._Y.backward(torch.as_tensor(dY, dtype=torch.float64))
        dX = self.X_cache.grad.detach().clone()
        with torch.no_grad():
            self.layer.weight -= lr * self.layer.weight.grad
            self.layer.bias -= lr * self.layer.bias.grad
        self.layer.zero_grad()
        self.W = self.layer.weight.detach().T.clone()
        self.b = self.layer.bias.detach().clone()
        return dX


In [ ]:
# exports: Y_dense, dX_dense, W_dense_step, b_dense_step
_rng_eq = np.random.default_rng(1456)
X_dense_eq = _rng_eq.normal(size=(4, 2, 4, 4))
dY_dense_eq = _rng_eq.normal(size=(4, 3))
_relu_eq = ReLULayer()
_flat_eq = Flatten()
_fc_eq = DenseLayer(32, 3, rng=np.random.default_rng(88))
Y_dense = _fc_eq.forward(_flat_eq.forward(_relu_eq.forward(X_dense_eq)))
dX_dense = _relu_eq.backward(_flat_eq.backward(_fc_eq.backward(dY_dense_eq, lr=0.1)))
W_dense_step = _fc_eq.W.detach().clone()
b_dense_step = _fc_eq.b.detach().clone()
print("head out:", tuple(Y_dense.shape), "-> dX:", tuple(dX_dense.shape))


In [ ]:
assert tuple(_fc_eq.layer.weight.shape) == (3, 32), \
    "nn.Linear keeps (d_out, d_in) — the transposition is real"
_W0_eq = DenseLayer(32, 3, rng=np.random.default_rng(88)).W
_expect = (torch.as_tensor(dY_dense_eq) @ _W0_eq.T).reshape(4, 2, 4, 4) \
    * (torch.as_tensor(X_dense_eq) > 0)
assert torch.allclose(dX_dense, _expect, atol=1e-12), \
    "dX = (dY @ W.T) reshaped, masked by the ReLU"
assert torch.allclose(b_dense_step, -0.1 * torch.as_tensor(dY_dense_eq).sum(dim=0),
                      atol=1e-12), "one SGD step on the bias"


## 14_simple_cnn

All the layers, end to end, trained.

### torch

The same architecture as one differentiable expression over four leaf tensors; `logits.backward(dS)` replaces the hand-written backward chain through dense, flatten, pool, ReLU and conv. The loss gradient `dS` is still derived by hand, so autograd begins exactly where the notebook's derivation ends. **What torch adds:** a five-layer chain rule in one line, and a training run that reproduces the looped NumPy net's numbers in a fraction of the time.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# hints:
# 1. Parameters are four leaf tensors; forward is conv2d → relu → max_pool2d → matmul.
# 2. logits.backward(dS) replaces the whole hand-written backward chain.
# 3. Draw conv weights before dense weights — the rng order is part of the init.
# 4. Keep dS derived by hand: autograd starts where the loss gradient enters.
# 5. reshape(batch, -1) flattens (C, H, W) in the same C-order as NumPy.


def stable_softmax(S):
    """Row-wise softmax with numerical stability."""
    S = torch.as_tensor(S, dtype=torch.float64)
    S_max = S.max(dim=1, keepdim=True).values
    E = torch.exp(S - S_max)
    return E / E.sum(dim=1, keepdim=True)


def cross_entropy_loss(logits, y):
    """Cross-entropy from raw logits — the gradient is still derived by hand."""
    P = stable_softmax(logits)
    n = len(y)
    eps = 1e-12
    y_idx = torch.as_tensor(np.asarray(y), dtype=torch.long)
    loss = float(-torch.mean(torch.log(P[torch.arange(n), y_idx] + eps)))
    dS = P.clone()
    dS[torch.arange(n), y_idx] -= 1.0
    dS /= n
    return loss, dS, P


class SimpleCNN:
    """Conv(3x3, 4 filters) → ReLU → MaxPool(2) → Flatten → Dense(n_classes).

    One differentiable expression over four leaf tensors; autograd replaces
    every hand-written backward method of the scratch net.
    """

    def __init__(self, img_size=8, n_classes=2, n_filters=4, rng=None):
        if rng is None:
            rng = np.random.default_rng(42)
        # Same draw order as the scratch net: conv weights first, then dense.
        conv_scale = np.sqrt(2.0 / (1 * 3 * 3))
        self.W_conv = torch.as_tensor(rng.normal(0, conv_scale, size=(n_filters, 1, 3, 3)))
        self.W_conv.requires_grad_(True)
        self.b_conv = torch.zeros(n_filters, dtype=torch.float64, requires_grad=True)
        pool_out = (img_size - 2) // 2
        dense_in = n_filters * pool_out * pool_out
        dense_scale = np.sqrt(2.0 / dense_in)
        self.W_dense = torch.as_tensor(rng.normal(0, dense_scale, size=(dense_in, n_classes)))
        self.W_dense.requires_grad_(True)
        self.b_dense = torch.zeros(n_classes, dtype=torch.float64, requires_grad=True)
        self.params = [self.W_conv, self.b_conv, self.W_dense, self.b_dense]

    def forward(self, X):
        """Forward pass through all layers — one differentiable expression."""
        X = torch.as_tensor(X, dtype=torch.float64)
        out = F.relu(F.conv2d(X, self.W_conv, self.b_conv))
        out = F.max_pool2d(out, 2)
        out = out.reshape(out.shape[0], -1)
        self._logits = out @ self.W_dense + self.b_dense
        return self._logits.detach()

    def backward(self, dS, lr):
        """Backward pass: seed autograd with dS, then one SGD step."""
        self._logits.backward(torch.as_tensor(dS, dtype=torch.float64))
        with torch.no_grad():
            for p in self.params:
                p -= lr * p.grad
        for p in self.params:
            p.grad = None

    def predict(self, X):
        logits = self.forward(X)
        return logits.argmax(dim=1).numpy()

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: loss_tail, logits_val, pred_val, val_acc
def _make_bar_data_eq(n, img_size=8, rng=None):
    X = np.zeros((n, 1, img_size, img_size))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        label = rng.integers(0, 2)
        y[i] = label
        noise = rng.normal(0, 0.1, size=(img_size, img_size))
        if label == 0:  # vertical bar
            X[i, 0, :, rng.integers(1, img_size - 1)] = 1.0
        else:  # horizontal bar
            X[i, 0, rng.integers(1, img_size - 1), :] = 1.0
        X[i, 0] += noise
    return X, y


X_train_eq, y_train_eq = _make_bar_data_eq(200, rng=np.random.default_rng(42))
X_val_eq, y_val_eq = _make_bar_data_eq(50, rng=np.random.default_rng(43))

cnn = SimpleCNN(img_size=8, n_classes=2, n_filters=4, rng=np.random.default_rng(42))
_losses_eq = []
for _epoch in range(30):
    _perm = np.random.default_rng(42 + _epoch).permutation(200)
    _epoch_loss = 0.0
    for _start in range(0, 200, 20):
        _idx = _perm[_start:_start + 20]
        _loss, _dS, _ = cross_entropy_loss(cnn.forward(X_train_eq[_idx]), y_train_eq[_idx])
        cnn.backward(_dS, 0.05)
        _epoch_loss += _loss
    _losses_eq.append(_epoch_loss / 10)

loss_tail = [float(v) for v in _losses_eq[-5:]]
logits_val = cnn.forward(X_val_eq[:6])
pred_val = [int(p) for p in cnn.predict(X_val_eq[:20])]
val_acc = float(cnn.score(X_val_eq, y_val_eq))
print(f"loss[-1] = {loss_tail[-1]:.4f}, val acc = {val_acc:.1%}")


In [ ]:
assert _losses_eq[-1] < 0.5 * _losses_eq[0], "the loss came down substantially"
assert val_acc >= 0.9, "the CNN separates vertical from horizontal bars"

_loss_chk, _dS_chk, _P_chk = cross_entropy_loss(cnn.forward(X_val_eq[:8]), y_val_eq[:8])
assert torch.allclose(_P_chk.sum(dim=1), torch.ones(8, dtype=torch.float64),
                      atol=1e-12), "softmax rows are distributions"
assert torch.allclose(_dS_chk.sum(dim=1), torch.zeros(8, dtype=torch.float64),
                      atol=1e-12), "softmax+CE gradient rows sum to zero"
assert [int(a) for a in logits_val.argmax(dim=1)] == pred_val[:6], \
    "predict is the argmax of the logits"


### library

`nn.Sequential` of the five standard modules, `F.cross_entropy` in place of the hand-fused softmax + NLL, and `torch.optim.SGD` applying the update. The initial weights are the same NumPy draw — conv first, then dense, transposed for `nn.Linear` — so the whole 30-epoch trajectory reproduces the scratch run. **What the library adds:** the training loop's three rituals — `zero_grad`, `backward`, `step`.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# hints:
# 1. F.cross_entropy fuses stable softmax + NLL; autograd rederives dS = (P−onehot)/n.
# 2. optim.SGD steps every parameter; take the lr from the backward call's argument.
# 3. Copy the conv weight directly and the dense weight .T — Linear stores (out, in).
# 4. Same rng draw order as the scratch init: conv first, then dense.


def stable_softmax(S):
    """torch.softmax — the max-subtraction trick is built in."""
    return torch.softmax(torch.as_tensor(S, dtype=torch.float64), dim=1)


def cross_entropy_loss(logits, y):
    """F.cross_entropy for the loss; autograd rederives the (P − onehot)/n gradient."""
    y_idx = torch.as_tensor(np.asarray(y), dtype=torch.long)
    S = torch.as_tensor(logits, dtype=torch.float64).detach().clone().requires_grad_(True)
    loss = F.cross_entropy(S, y_idx)
    loss.backward()
    P = stable_softmax(S.detach())
    return float(loss), S.grad.detach().clone(), P


class SimpleCNN:
    """The same net as five standard modules, stepped by torch.optim.SGD."""

    def __init__(self, img_size=8, n_classes=2, n_filters=4, rng=None):
        if rng is None:
            rng = np.random.default_rng(42)
        pool_out = (img_size - 2) // 2
        dense_in = n_filters * pool_out * pool_out
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(1, n_filters, 3),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Flatten(),
            torch.nn.Linear(dense_in, n_classes),
        ).double()
        conv_scale = np.sqrt(2.0 / (1 * 3 * 3))
        dense_scale = np.sqrt(2.0 / dense_in)
        with torch.no_grad():
            W_conv0 = rng.normal(0, conv_scale, size=(n_filters, 1, 3, 3))
            self.net[0].weight.copy_(torch.as_tensor(W_conv0))
            self.net[0].bias.zero_()
            W_dense0 = rng.normal(0, dense_scale, size=(dense_in, n_classes))
            self.net[4].weight.copy_(torch.as_tensor(W_dense0).T)
            self.net[4].bias.zero_()
        self.opt = torch.optim.SGD(self.net.parameters(), lr=0.05)

    def forward(self, X):
        """One module call; the graph is kept for backward."""
        self._logits = self.net(torch.as_tensor(X, dtype=torch.float64))
        return self._logits.detach()

    def backward(self, dS, lr):
        """zero_grad → backward(dS) → step: the library's three rituals."""
        for group in self.opt.param_groups:
            group["lr"] = lr
        self.opt.zero_grad()
        self._logits.backward(torch.as_tensor(dS, dtype=torch.float64))
        self.opt.step()

    def predict(self, X):
        return self.forward(X).argmax(dim=1).numpy()

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: loss_tail, logits_val, pred_val, val_acc
def _make_bar_data_eq(n, img_size=8, rng=None):
    X = np.zeros((n, 1, img_size, img_size))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        label = rng.integers(0, 2)
        y[i] = label
        noise = rng.normal(0, 0.1, size=(img_size, img_size))
        if label == 0:  # vertical bar
            X[i, 0, :, rng.integers(1, img_size - 1)] = 1.0
        else:  # horizontal bar
            X[i, 0, rng.integers(1, img_size - 1), :] = 1.0
        X[i, 0] += noise
    return X, y


X_train_eq, y_train_eq = _make_bar_data_eq(200, rng=np.random.default_rng(42))
X_val_eq, y_val_eq = _make_bar_data_eq(50, rng=np.random.default_rng(43))

cnn = SimpleCNN(img_size=8, n_classes=2, n_filters=4, rng=np.random.default_rng(42))
_losses_eq = []
for _epoch in range(30):
    _perm = np.random.default_rng(42 + _epoch).permutation(200)
    _epoch_loss = 0.0
    for _start in range(0, 200, 20):
        _idx = _perm[_start:_start + 20]
        _loss, _dS, _ = cross_entropy_loss(cnn.forward(X_train_eq[_idx]), y_train_eq[_idx])
        cnn.backward(_dS, 0.05)
        _epoch_loss += _loss
    _losses_eq.append(_epoch_loss / 10)

loss_tail = [float(v) for v in _losses_eq[-5:]]
logits_val = cnn.forward(X_val_eq[:6])
pred_val = [int(p) for p in cnn.predict(X_val_eq[:20])]
val_acc = float(cnn.score(X_val_eq, y_val_eq))
print(f"loss[-1] = {loss_tail[-1]:.4f}, val acc = {val_acc:.1%}")


In [ ]:
# autograd through F.cross_entropy reproduces the hand-derived gradient.
_loss_chk, _dS_chk, _P_chk = cross_entropy_loss(cnn.forward(X_val_eq[:8]), y_val_eq[:8])
_manual_eq = _P_chk.clone()
_manual_eq[torch.arange(8), torch.as_tensor(np.asarray(y_val_eq[:8]), dtype=torch.long)] -= 1.0
_manual_eq /= 8
assert torch.allclose(_dS_chk, _manual_eq, atol=1e-12), "autograd rederives (P − onehot)/n"

assert val_acc >= 0.9, "same task, same result through torch.optim"
assert _losses_eq[-1] < 0.5 * _losses_eq[0], "the loss came down substantially"
